# 03 · ESP32 export and fidelity checks

**Generates** in `results/esp32_export/<dataset>/`:
* `esp32_dt_test/esp32_dt_model.h`: the tree as C arrays (exact export)
* `esp32_dt_test/esp32_test_vectors.h`: held-out samples (NaN/Inf-safe) with the Python predictions
* `esp32_dt_test/esp32_dt_test.ino`: benchmark sketch that times only `predict_esp32_dt()` and prints after timing
* `esp32_dt_footprint/…`: sketch pair for measuring the model's own flash cost
* `python_expected_predictions.csv`, `export_manifest.json`, `c_logic_fidelity.json`, `raw_threshold_check.json`

**Exact export.** Every threshold is mapped into the raw float32 input space by bisection against the fitted pipeline, so the device performs no preprocessing arithmetic
and agrees with scikit-learn for every float32 input. `raw_threshold_check.json` records the per-node check. The legacy export (float32 scaling on the device,
nearest-float32 thresholds) is generated alongside for comparison.

**Checks that need no hardware**
1. numpy re-implementation of the C code vs scikit-learn on the entire held-out test split;
2. the generated header compiled with a host C compiler vs scikit-learn on the flashed vectors and on the entire test split.

If the compiler sits in a folder whose path contains a space (Windows), set `FAITHTREE_CC` to the full path of `gcc.exe`.

**Timing.** The sketch times only the prediction call. Printing one result line at 115 200 baud costs about 1.4 ms, so timing a loop that also prints measures the serial port, not the model.

In [ ]:
# ---- Setup: find the package root, work inside results/, import the shared code ----
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "framework.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Open this notebook from inside the paper_package folder.")
    ROOT = ROOT.parent
os.environ["PAPER_PACKAGE_ROOT"] = str(ROOT)

# >>> WHICH DATASET?  nf_ton_iot_v2 | cic_ton_iot | nf_unsw_nb15_v3
# run_all.py sets ACTIVE_DATASET; when running by hand, edit the default below
os.environ.setdefault("ACTIVE_DATASET", "nf_unsw_nb15_v3")

sys.path.insert(0, str(ROOT / "src"))
(ROOT / "results").mkdir(exist_ok=True)
os.chdir(ROOT / "results")            # every output (csv / png / pkl / headers) lands under results/

from run_settings import *            # PAPER_MODE: full | quick | smoke

In [ ]:
from eval_helpers import *
from framework import _json_default, _normalize_col
from esp32_export import *
import esp32_export as ex
print('Dataset:', ACTIVE_DATASET, '| samples to flash:', ESP32_N_SAMPLES, '| mode:', ESP32_SAMPLE_MODE)

In [ ]:
artifact, metadata = load_artifact_and_metadata()
ARTIFACT_PATH = metadata["model_path"]
saved_pipeline = artifact["pipeline"]
label_encoder = artifact["label_encoder"]

X, y, le_check, numeric_cols_check = load_publishability_data()
X_train, X_val, X_test, y_train, y_val, y_test, X_final, y_final = same_split(X, y)
class_names = list(le_check.classes_)

print("Loaded data:", X.shape, "| test:", X_test.shape)
print("Classes:", class_names)
print("Best k:", metadata.get("best_k"))
print("Best params:", metadata.get("best_params"))


print('loaded', X.shape, 'test', X_test.shape, '| K =', metadata['best_k'])

## 1 · Choose which held-out samples go on the board

In [ ]:
OUT_DIR = ESP32_DIR / "esp32_dt_test"
OUT_DIR.mkdir(parents=True, exist_ok=True)

sel = select_validation_samples(y_test, ESP32_N_SAMPLES, ESP32_SAMPLE_MODE, SEED)
X_sample, y_sample = X_test.iloc[sel], np.asarray(y_test)[sel]
py_pred = saved_pipeline.predict(X_sample)

cnt = pd.Series(y_sample).map(dict(enumerate(class_names))).value_counts().rename("flashed samples")
print(f"{len(sel)} samples selected ({ESP32_SAMPLE_MODE}); {cnt.size}/{len(class_names)} classes present")
display(cnt.to_frame())
print("Python accuracy on the flashed subset:", round(float((py_pred == y_sample).mean()), 4))

## 2 · Write header, test vectors, sketches

In [ ]:
model_header = export_tree_pipeline_to_c(artifact, OUT_DIR)
vec_header   = create_test_vectors_header(X_sample, y_sample, py_pred, artifact, OUT_DIR, sample_ids=X_sample.index.values)
bench_ino    = write_benchmark_sketch(OUT_DIR)
foot_ino     = write_footprint_sketch(model_header, ESP32_DIR)

exp = pd.concat([pd.DataFrame({"sample_id": X_sample.index.values, "true_label": y_sample, "python_pred": py_pred}),
                 X_sample[artifact["numeric_cols"]].reset_index(drop=True)], axis=1)
exp.to_csv(ESP32_DIR / "python_expected_predictions.csv", index=False)

for p in (model_header, vec_header, bench_ino, foot_ino):
    print(f"{p.name:28s} {p.stat().st_size/1024:10.1f} KB   {p}")

# ---- flash budget sanity check (default ESP32 app partition = 1,310,720 B) ----
n_nodes = artifact["pipeline"].named_steps["clf"].tree_.node_count
est_model = n_nodes * (2 + 2 + 1 + 4 + 1) + 4 * len(artifact["numeric_cols"])   # left, right, feature, threshold, class + medians
est_vec = len(sel) * (len(artifact["numeric_cols"]) * 4 + 6)
est_total = 300_000 + est_model + est_vec          # ~300 kB = Arduino core + Serial (calibrated on earlier builds)
print(f"\nestimated firmware: model {est_model/1024:.0f} KB + vectors {est_vec/1024:.0f} KB (+core) ≈ {est_total/1310720:.0%} of the 1.25 MB app partition")
if est_total > 0.90 * 1310720:
    print("WARNING: close to / over the flash limit -> lower ESP32_N_SAMPLES (e.g. 3000) or choose a larger partition scheme in the IDE")

## 3 · Fidelity checks without hardware

In [ ]:
# (a) numpy re-implementation of the C logic on the ENTIRE held-out test set
#     exact export = raw-space thresholds -> must agree on every row; the legacy export is shown for comparison
py_full  = saved_pipeline.predict(X_test)
emu_full = emulate_c_predict(artifact, X_test)
leg_full = emulate_c_predict_legacy(artifact, X_test)
agree = emu_full == py_full

fid = {"export_version": 3,
       "raw_threshold_check": json.loads((OUT_DIR / "raw_threshold_check.json").read_text()),
       "n_test": int(len(py_full)),
       "c_logic_vs_sklearn_matches": int(agree.sum()),
       "c_logic_vs_sklearn_agreement": float(agree.mean()),
       "legacy_export_matches": int((leg_full == py_full).sum())}

# rows where any C logic disagrees with scikit-learn (expected: none for the exact export)
mismatch_report(artifact, X_test, py_full, emu_full, y_test, out_csv=ESP32_DIR / "c_logic_mismatches_v3.csv")
leg = mismatch_report(artifact, X_test, py_full, leg_full, y_test, out_csv=ESP32_DIR / "c_logic_mismatches_legacy.csv")
if len(leg):
    leg["true"] = leg["true"].map(dict(enumerate(class_names)))
    print("Legacy-export disagreements (now fixed):"); display(leg)

# (b) the generated header compiled with the host C compiler: flashed vectors AND the full test set
host = host_c_check(OUT_DIR, py_pred)
fid["host_c_flashed_vectors"] = host if host is not None else "no C compiler found (set FAITHTREE_CC to gcc.exe)"
full = host_c_check_full(OUT_DIR, artifact, X_test, py_full)
if full is None:
    fid["host_c_full_test_set"] = "no C compiler found (set FAITHTREE_CC to gcc.exe)"
else:
    fid["host_c_full_test_set"] = {k: v for k, v in full.items() if k != "c_pred"}

(ESP32_DIR / "c_logic_fidelity.json").write_text(json.dumps(fid, indent=2, default=str))
print(json.dumps(fid, indent=2, default=str))

if agree.all():
    print(f"\nC logic reproduces scikit-learn on all {len(py_full):,} held-out rows "
          f"(legacy export: {fid['legacy_export_matches']:,})")
else:
    print(f"\nWARNING: {int((~agree).sum())} mismatches -- see c_logic_mismatches_v3.csv before flashing")


In [ ]:
manifest = {"dataset": ACTIVE_DATASET, "sample_mode": ESP32_SAMPLE_MODE, "n_flashed_samples": int(len(sel)),
            "classes_present": int(cnt.size), "n_classes": len(class_names), "n_nodes": int(n_nodes),
            "n_selected_features": int(len(artifact["pipeline"].named_steps["selector"].get_support(indices=True))),
            "n_raw_features": len(artifact["numeric_cols"]), "model_header_kb": round(model_header.stat().st_size/1024, 1),
            "python_accuracy_on_flashed_subset": float((py_pred == y_sample).mean()), "model_pkl": ARTIFACT_PATH}
(ESP32_DIR / "export_manifest.json").write_text(json.dumps(manifest, indent=2))
print(f'''
================  WHAT TO DO ON THE BOARD  ================
1. Arduino IDE: open  {OUT_DIR / 'esp32_dt_test.ino'}
   (board: ESP32 Dev Module, arduino-esp32 core >= 2.0, Serial Monitor 115200 baud)
2. Upload, open the Serial Monitor, wait for "#DONE", save ALL the output to
   {ROOT / 'esp32' / 'serial_logs' / (ACTIVE_DATASET + '_serial.txt')}
   (or:  python tools/capture_serial.py --port COM5 --dataset {ACTIVE_DATASET})
3. Optional, for the model-only flash number: compile {ESP32_DIR / 'esp32_dt_footprint'} twice
   (USE_MODEL 1 and 0) and save the two serial outputs as
   esp32/serial_logs/{ACTIVE_DATASET}_footprint_with_model.txt / _without_model.txt
4. Run notebook 05.
===========================================================
''')